In [ ]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
data = pd.read_csv(r"/content/diabetes.csv")

glucose_values = data["Glucose"].values.astype(float)
bmi_values     = data["BMI"].values.astype(float)
age_values     = data["Age"].values.astype(float)

# Normalise features to [0, 1] for the neural network
glucose_min, glucose_max = glucose_values.min(), glucose_values.max()
bmi_min,     bmi_max     = bmi_values.min(),     bmi_values.max()
age_min,     age_max     = age_values.min(),      age_values.max()

def normalise(glucose, bmi, age):
    g = (glucose - glucose_min) / (glucose_max - glucose_min + 1e-8)
    b = (bmi     - bmi_min)     / (bmi_max     - bmi_min     + 1e-8)
    a = (age      - age_min)     / (age_max     - age_min     + 1e-8)
    return np.array([g, b, a], dtype=np.float32)

In [ ]:
class DQN(nn.Module):
    """
    3-layer fully-connected network.
    Input  : state vector (3 features)
    Output : Q-value for each action (3 actions)
    """
    def __init__(self, state_dim=3, action_dim=3, hidden=128):
        super(DQN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class ReplayBuffer:
    """Experience Replay: stores (s, a, r, s', done) tuples."""
    def __init__(self, capacity=10_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(np.array(states)),
            torch.LongTensor(actions),
            torch.FloatTensor(rewards),
            torch.FloatTensor(np.array(next_states)),
            torch.FloatTensor(dones)
        )

    def __len__(self):
        return len(self.buffer)

In [ ]:
class DQNAgent:
    def __init__(
        self,
        state_dim=3,
        action_dim=3,
        lr=1e-3,
        gamma=0.9,
        epsilon=1.0,
        epsilon_min=0.05,
        epsilon_decay=0.995,
        batch_size=64,
        target_update_freq=100,
        buffer_capacity=10_000
    ):
        self.action_dim        = action_dim
        self.gamma             = gamma
        self.epsilon           = epsilon
        self.epsilon_min       = epsilon_min
        self.epsilon_decay     = epsilon_decay
        self.batch_size        = batch_size
        self.target_update_freq = target_update_freq
        self.step_count        = 0

        # Online network  → trained every step
        # Target network  → updated periodically (stable targets)
        self.online_net = DQN(state_dim, action_dim)
        self.target_net = DQN(state_dim, action_dim)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.online_net.parameters(), lr=lr)
        self.loss_fn   = nn.MSELoss()
        self.replay    = ReplayBuffer(buffer_capacity)

    # ── ε-greedy action selection ──
    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        state_t = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            q_vals = self.online_net(state_t)
        return q_vals.argmax().item()

    # ── Store experience ──
    def store(self, state, action, reward, next_state, done):
        self.replay.push(state, action, reward, next_state, done)

    # ── Learn from a mini-batch ──
    def learn(self):
        if len(self.replay) < self.batch_size:
            return None                    # wait until buffer is ready

        states, actions, rewards, next_states, dones = self.replay.sample(self.batch_size)

        # Current Q-values from online network
        current_q = self.online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Target Q-values from target network  (Bellman equation)
        with torch.no_grad():
            max_next_q = self.target_net(next_states).max(1)[0]
            target_q   = rewards + self.gamma * max_next_q * (1 - dones)

        loss = self.loss_fn(current_q, target_q)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Decay epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

        # Sync target network periodically
        self.step_count += 1
        if self.step_count % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.online_net.state_dict())

        return loss.item()

In [ ]:
dose_effect = [2.0, 5.0, 8.0]   # glucose reduction per action

def reward_function(glucose):
    """Reward shaping: encourage normal glucose range (80–120)."""
    if 80 <= glucose <= 120:
        return +10.0
    elif 70 <= glucose <= 140:
        return +2.0
    else:
        return -5.0

def step(glucose, bmi, age, action):
    """Apply action, add stochasticity, return (next_state, reward, done)."""
    glucose -= dose_effect[action]          # insulin reduces glucose
    glucose += random.uniform(0, 3)         # natural variation
    glucose  = max(40.0, min(glucose, 200.0))

    reward     = reward_function(glucose)
    done       = False                      # continuous task; never terminates early
    next_state = normalise(glucose, bmi, age)
    return next_state, reward, done, glucose


In [ ]:
EPISODES    = 3000
STEPS_EP    = 30       # steps per episode (same as SARSA notebook)

agent = DQNAgent()

reward_history = []
loss_history   = []

print("=" * 55)
print("       DQN Training – Diabetes Dosage Control")
print("=" * 55)

for ep in range(EPISODES):
    idx     = random.randint(0, len(glucose_values) - 1)
    glucose = glucose_values[idx]
    bmi     = bmi_values[idx]
    age     = age_values[idx]

    state       = normalise(glucose, bmi, age)
    total_reward = 0.0
    ep_losses    = []

    for _ in range(STEPS_EP):
        action                           = agent.select_action(state)
        next_state, reward, done, glucose = step(glucose, bmi, age, action)

        agent.store(state, action, reward, next_state, float(done))
        loss = agent.learn()

        if loss is not None:
            ep_losses.append(loss)

        state        = next_state
        total_reward += reward

    reward_history.append(total_reward)
    if ep_losses:
        loss_history.append(np.mean(ep_losses))

    if ep % 500 == 0:
        avg_r = np.mean(reward_history[-50:]) if len(reward_history) >= 50 else np.mean(reward_history)
        print(f"  Episode {ep:>4d} | Avg Reward (last 50): {avg_r:>7.2f} | ε: {agent.epsilon:.3f}")

print("\nTraining Complete!")

In [ ]:
def moving_average(data, window=50):
    return np.convolve(data, np.ones(window) / window, mode='valid')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("DQN Training – Diabetes Insulin Dosage", fontsize=14, fontweight='bold')

# --- Plot 1: Raw reward ---
axes[0].plot(reward_history, alpha=0.4, color='steelblue', label='Episode Reward')
axes[0].plot(moving_average(reward_history), color='navy', linewidth=2, label='Moving Avg (50)')
axes[0].set_title("Reward Learning Curve")
axes[0].set_xlabel("Episodes")
axes[0].set_ylabel("Total Reward")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Plot 2: Loss ---
if loss_history:
    axes[1].plot(loss_history, alpha=0.5, color='tomato', label='MSE Loss')
    axes[1].plot(moving_average(loss_history, 30), color='darkred', linewidth=2, label='Moving Avg (30)')
    axes[1].set_title("Training Loss (MSE)")
    axes[1].set_xlabel("Episodes")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

# --- Plot 3: Final 500 episode distribution ---
final_rewards = reward_history[-500:]
axes[2].hist(final_rewards, bins=30, color='mediumseagreen', edgecolor='black', alpha=0.8)
axes[2].axvline(np.mean(final_rewards), color='red', linestyle='--', linewidth=2, label=f'Mean={np.mean(final_rewards):.1f}')
axes[2].set_title("Reward Distribution (last 500 eps)")
axes[2].set_xlabel("Total Reward")
axes[2].set_ylabel("Frequency")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("DQN_training_results.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
print("\n" + "=" * 45)
print("       FINAL PERFORMANCE SUMMARY (DQN)")
print("=" * 45)
print(f"  Overall Avg Reward           : {np.mean(reward_history):.2f}")
print(f"  Final   Avg Reward (last 100): {np.mean(reward_history[-100:]):.2f}")
print(f"  Best Episode Reward          : {np.max(reward_history):.2f}")
print(f"  Final Epsilon (exploration)  : {agent.epsilon:.4f}")

In [ ]:
action_names = ["Low Dose", "Medium Dose", "High Dose"]
action_counts = {a: 0 for a in action_names}

agent.online_net.eval()
with torch.no_grad():
    for g, b, a in zip(glucose_values, bmi_values, age_values):
        s     = normalise(g, b, a)
        s_t   = torch.FloatTensor(s).unsqueeze(0)
        q_val = agent.online_net(s_t)
        act   = q_val.argmax().item()
        action_counts[action_names[act]] += 1

print("\n  Greedy Policy – Action Distribution over Dataset:")
for name, count in action_counts.items():
    pct = count / len(glucose_values) * 100
    print(f"    {name:>12s}: {count:>4d} patients ({pct:.1f}%)")


In [ ]:
def predict_dosage(glucose, bmi, age):
    state = normalise(glucose, bmi, age)
    state_t = torch.FloatTensor(state).unsqueeze(0)
    agent.online_net.eval()
    with torch.no_grad():
        q_values   = agent.online_net(state_t).squeeze().numpy()
    best_action = int(np.argmax(q_values))

    print("\n" + "=" * 35)
    print("      DQN DOSAGE PREDICTION")
    print("=" * 35)
    print(f"  Glucose Level : {glucose}")
    print(f"  BMI           : {bmi}")
    print(f"  Age           : {age}")
    print(f"  Q-values      : Low={q_values[0]:.2f} | Med={q_values[1]:.2f} | High={q_values[2]:.2f}")
    print(f"  Recommended   : {action_names[best_action]}")
    new_glucose = glucose - dose_effect[best_action]
    print(f"  Est. Glucose after dose : {new_glucose:.1f}")

# ── Interactive loop ──
while True:
    print("\nEnter Patient Details")
    try:
        glucose = float(input("Enter Glucose Level : "))
        bmi     = float(input("Enter BMI           : "))
        age     = float(input("Enter Age           : "))
        predict_dosage(glucose, bmi, age)
    except ValueError:
        print("Invalid input. Please enter numeric values.")

    choice = input("\nTest another patient? (yes/no): ")
    if choice.strip().lower() != "yes":
        print("\nSimulation Ended.")
        break